- VYOM PATEL
- 202307020084
- BTECH CSE AIML SEM 7
- AAI LAB EXPERIMENT 4

# Building a RAG Knowledge Base with LangChain, ChromaDB/FAISS & the Gemini API




## 0. Setup: install libraries


In [4]:
!pip install -q -U \
    langchain \
    langchain-community \
    langchain-core \
    langchain-text-splitters \
    langchain-google-genai \
    langchain-chroma \
    chromadb \
    faiss-cpu \
    pypdf


### Provide your Gemini API key



In [5]:
import os

api_key = None

try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    pass

if not api_key:
    from getpass import getpass
    api_key = getpass("Enter your Gemini API key (from https://aistudio.google.com/app/apikey): ")

os.environ["GEMINI_API_KEY"] = api_key
print("Gemini API key is set." if os.environ.get("GEMINI_API_KEY") else "No key set!")


Gemini API key is set.


## Step 1: Find the LangChain lib for converting text to embeddings



In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"],
)

test_vector = embedding_model.embed_query("What is retrieval-augmented generation?")
print("Embedding length:", len(test_vector))
print("First 5 values:", test_vector[:5])


Embedding length: 3072
First 5 values: [-0.020411316, 0.007700572, -0.0019213042, -0.0767533, -0.010083337]


## Step 2: Define our data source



In [8]:
# Option A: create a small sample knowledge base of .txt files
import os

KB_DIR = "knowledge_base"
os.makedirs(KB_DIR, exist_ok=True)

sample_docs = {
"01_what_is_rag.txt": '''Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system
with a large language model (LLM). Instead of relying only on knowledge baked into the
model during training, a RAG system first searches an external knowledge base for
relevant text, then passes that text to the LLM as context so it can generate a
grounded, up-to-date answer.

RAG reduces hallucination, lets a model answer questions about private or recent
documents, and avoids retraining the model whenever the underlying data changes. A
typical RAG pipeline has two phases: an indexing phase (split documents into chunks,
embed them, store in a vector database) and a query phase (embed the user's question,
retrieve the closest chunks, feed them to the LLM with the question).''',

"02_langchain_overview.txt": '''LangChain Overview

LangChain is a framework for building LLM-powered applications. For a RAG use case, the
most relevant building blocks are: document loaders (TextLoader, PyPDFLoader), text
splitters (RecursiveCharacterTextSplitter), embedding classes (GoogleGenerativeAIEmbeddings,
OpenAIEmbeddings, HuggingFaceEmbeddings), vector store integrations (Chroma, FAISS), and
retrievers, which expose a simple interface a chain or agent can call to fetch relevant
context.''',

"03_vector_databases.txt": '''Vector Databases: FAISS and Chroma

A vector database stores embeddings (arrays of floating point numbers) alongside the
original text and metadata, and supports similarity search - finding the stored vectors
closest to a query vector, usually via cosine similarity.

FAISS (Facebook AI Similarity Search) is a library from Meta for efficient similarity
search over large collections of dense vectors; it is lightweight and great for local
prototypes. Chroma (ChromaDB) is an open-source embedding database built for LLM apps,
with built-in persistence and metadata filtering, and integrates directly with LangChain
via langchain-chroma.''',

"04_embeddings.txt": '''Text Embeddings

An embedding is a numeric vector representation of text such that semantically similar
texts end up close together in vector space. Embedding models are trained so that
semantic similarity is reflected as geometric closeness (e.g. cosine similarity).

Gemini's text-embedding-004 model, OpenAI's text-embedding-3-small, and local
sentence-transformers models are all common choices. Once documents are embedded and
stored, a user query is embedded with the SAME model, and the vector database returns
the stored chunks whose embeddings are closest to the query embedding.''',

"05_agents.txt": '''AI Agents

An AI agent uses a large language model as a "reasoning engine" to decide which actions
to take, in what order, to accomplish a goal. Agents have access to tools (a web search
tool, a calculator, a retriever) and can call them, observe results, and decide the next
step.

RAG is often exposed to an agent as a single tool: "search the knowledge base." The
agent decides when a question needs a lookup versus when it can answer directly, calls
the retriever tool, reads the returned chunks, and composes its final answer.''',
}

for filename, content in sample_docs.items():
    with open(os.path.join(KB_DIR, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"Created {len(sample_docs)} sample files in '{KB_DIR}/'")


Created 5 sample files in 'knowledge_base/'


In [9]:
# Option B : upload your own .txt / .pdf files into the same knowledge_base/ folder

from google.colab import files
uploaded = files.upload()
for fname in uploaded.keys():
    dest = os.path.join(KB_DIR, fname)
    os.rename(fname, dest)
    print("Added:", dest)


Saving 084Cost_Benefit_Analysis_EpiPredict_AI.pdf to 084Cost_Benefit_Analysis_EpiPredict_AI.pdf
Added: knowledge_base/084Cost_Benefit_Analysis_EpiPredict_AI.pdf


In [10]:
# Load every .txt (and .pdf, if you uploaded any) file in knowledge_base/, then split into chunks
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

txt_loader = DirectoryLoader(KB_DIR, glob="*.txt", loader_cls=TextLoader,
                              loader_kwargs={"encoding": "utf-8"})
documents = txt_loader.load()

# also pick up any PDFs you uploaded
pdf_loader = DirectoryLoader(KB_DIR, glob="*.pdf", loader_cls=PyPDFLoader)
documents += pdf_loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
chunks = splitter.split_documents(documents)

print(f"Loaded {len(documents)} document(s) -> split into {len(chunks)} chunks.")


/tmp/ipykernel_632/2792997006.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader


Loaded 9 document(s) -> split into 30 chunks.


## Step 3: Create a Chroma vector store instance



In [11]:
from langchain_chroma import Chroma

CHROMA_DIR = "chroma_store"

vectorstore = Chroma(
    collection_name="course_knowledge_base",
    embedding_function=embedding_model,
    persist_directory=CHROMA_DIR,
)
print("Chroma collection ready at:", CHROMA_DIR)


Chroma collection ready at: chroma_store


## Step 4: Convert text into embedded format and store it in Chroma



In [12]:
vectorstore.add_documents(chunks)
print(f"Indexed {len(chunks)} chunks into Chroma.")


Indexed 30 chunks into Chroma.


## Step 5: Retrieve data from the vector store based on a user query



In [13]:
def retrieve(query, k=3):
    return vectorstore.similarity_search(query, k=k)

user_query = "What is the difference between FAISS and Chroma?"
results = retrieve(user_query, k=3)

print(f"Query: {user_query}\n")
for i, doc in enumerate(results, start=1):
    source = os.path.basename(doc.metadata.get("source", "unknown"))
    print(f"--- Result {i} (source: {source}) ---")
    print(doc.page_content.strip()[:300])
    print()


Query: What is the difference between FAISS and Chroma?

--- Result 1 (source: 03_vector_databases.txt) ---
FAISS (Facebook AI Similarity Search) is a library from Meta for efficient similarity
search over large collections of dense vectors; it is lightweight and great for local
prototypes. Chroma (ChromaDB) is an open-source embedding database built for LLM apps,
with built-in persistence and metadata fi

--- Result 2 (source: 03_vector_databases.txt) ---
Vector Databases: FAISS and Chroma

A vector database stores embeddings (arrays of floating point numbers) alongside the
original text and metadata, and supports similarity search - finding the stored vectors
closest to a query vector, usually via cosine similarity.

--- Result 3 (source: 04_embeddings.txt) ---
Gemini's text-embedding-004 model, OpenAI's text-embedding-3-small, and local
sentence-transformers models are all common choices. Once documents are embedded and
stored, a user query is embedded with the SAME model, and the v

## Bonus: complete the RAG loop — generate an answer with Gemini



In [20]:
# Google Gemini API setup + LangChain RAG

import os


# If using Google Colab Secrets:
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

# Check that the key exists without exposing it
print("Gemini API key loaded:",
      bool(os.environ.get("GEMINI_API_KEY")))

# LangChain imports
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Current Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant answering questions using ONLY the context below.

If the answer isn't in the context, say you don't know.

Context:
{context}

Question:
{question}

Answer:
""")

def ask(question: str, k: int = 3) -> str:

    retrieved_docs = retrieve(question, k=k)

    context = "\n\n".join(
        f"[Source: {os.path.basename(d.metadata.get('source', 'unknown'))}]\n"
        f"{d.page_content}"
        for d in retrieved_docs
    )

    chain = rag_prompt | llm

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content


# Test
print(ask("What is the difference between FAISS and Chroma?"))

Gemini API key loaded: True
[{'type': 'text', 'text': 'Based on the provided context, the differences between FAISS and Chroma are:\n\n* **FAISS (Facebook AI Similarity Search)** is a lightweight library from Meta designed for efficient similarity search over large collections of dense vectors, making it great for local prototypes.\n* **Chroma (ChromaDB)** is an open-source embedding database built specifically for LLM apps. Unlike FAISS, it includes built-in persistence and metadata filtering, and it integrates directly with LangChain via `langchain-chroma`.', 'extras': {'signature': 'EpYQCpMQARFNMg+dl9Nq2utPxegopcE8f2fBBiNgYup5mqLc3QGdlZtJ1vcuVFlKf5XiLJUFisz/1yurRDpoao3r2eYpEiW1BICgkc/xBSW6PqWvpdUwrm41Rrcmk6z5ESsPIxRAbgsn/ChzAqKOwT6Iwv76fL5hnkb9zjH0gbKlxcVkMES8dGvP5aXyxH1bZXdfoPzKIGksvlEK+TH5ID1oWJk4lo0XsbrbQ7bfA+k5AqPloLTbORIynt3YWbOo8Qt4FRoq44lPctqZTfza26RtO7pb472FTJHAs/U057Zf/BhqbdyQMi2qLEpWtd/lvuypRlM5ros6OXfZ392GIvqvvXlKvaG9zAlNei2sJFW40xMfUUC4+GeQwYWUBq+i3zp6eChv4RvJcEu/D4Q28wv

In [21]:
# Try a few more questions against your knowledge base
for q in [
    "What does RAG stand for and why is it useful?",
    "How does an AI agent decide to use a retrieval tool?",
    "Which LangChain class would I use to load a PDF?",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)


Q: What does RAG stand for and why is it useful?
A: [{'type': 'text', 'text': 'Based on the provided context:\n\n**What RAG stands for:** \nRAG stands for **Retrieval-Augmented Generation**.\n\n**Why it is useful:**\n* It reduces hallucination.\n* It allows a model to answer questions about private or recent documents.\n* It avoids the need to retrain the model whenever the underlying data changes.\n* It enables the model to generate grounded, up-to-date answers by searching an external knowledge base instead of relying solely on knowledge baked in during training.', 'extras': {'signature': 'Eq8SCqwSARFNMg8jIPRXw5Z3nWlmGhlCyqxi/wZFAL7W2nu0syhiclFMLEqaEkAYHSfRmN+rHw37HiLT4QqH0aqjChX41UktZ3Zvfk8hhU49flGuynQCsXyI0fN8HKnynMBMp1PuSppSvp+TyOrpjyBRxbdCMSDG2GKiYlPW08i+X6/qCdIbNnfweQ9ZJ8zJEb98Kc8egbTGQ8n1e4H0jwnEPLRoISyrQ2FoTz2vzz8I73NWcQ1Uk8ZxAMj9M1hX5jwP/qiS2/lMmGRrGcX+vzpg4MwP74wvr0+zfCk+Gzv1XXHX+D2XfkeodnKnuTuNnRazrrnt0msjo2BRqIv6FcXNslyK97C//D/7Ic53MtZtn0d9rATh8qIECNEvFOkQ/Jiga4uV70kQIrUEE

## Alternative: swap in FAISS instead of Chroma



In [22]:
from langchain_community.vectorstores import FAISS

# Steps 3 + 4 collapse into one call for FAISS:
faiss_store = FAISS.from_documents(chunks, embedding_model)
faiss_store.save_local("faiss_store")

# Step 5, same as before:
faiss_results = faiss_store.similarity_search("How does an AI agent decide to use a retrieval tool?", k=3)
for i, doc in enumerate(faiss_results, start=1):
    print(f"--- Result {i} (source: {os.path.basename(doc.metadata.get('source','unknown'))}) ---")
    print(doc.page_content.strip()[:300])
    print()


--- Result 1 (source: 05_agents.txt) ---
RAG is often exposed to an agent as a single tool: "search the knowledge base." The
agent decides when a question needs a lookup versus when it can answer directly, calls
the retriever tool, reads the returned chunks, and composes its final answer.

--- Result 2 (source: 05_agents.txt) ---
AI Agents

An AI agent uses a large language model as a "reasoning engine" to decide which actions
to take, in what order, to accomplish a goal. Agents have access to tools (a web search
tool, a calculator, a retriever) and can call them, observe results, and decide the next
step.

--- Result 3 (source: 01_what_is_rag.txt) ---
Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system
with a large language model (LLM). Instead of relying only on knowledge baked into the
model during training, a RAG system first searches an external knowledge base for
relev



## Recap

| Step | What we did | LangChain / Gemini piece |
|---|---|---|
| 1 | Chose an embedding model | `GoogleGenerativeAIEmbeddings` (`text-embedding-004`) |
| 2 | Defined the data source | `DirectoryLoader` + `TextLoader`/`PyPDFLoader`, `RecursiveCharacterTextSplitter` |
| 3 | Created the vector store | `Chroma(...)` (or `FAISS.from_documents(...)`) |
| 4 | Embedded & stored chunks | `vectorstore.add_documents(chunks)` |
| 5 | Retrieved + generated an answer | `similarity_search()` + `ChatGoogleGenerativeAI` (`gemini-2.0-flash`) |

**Next steps you could try:**
- Replace the sample `knowledge_base/` files with your own course notes or PDFs (use the upload cell in Step 2).
- Add metadata filters (e.g. filter by source file) to `similarity_search`.
- Wrap `ask()` as a LangChain **tool** and give it to an **agent**, so the agent decides on its own when to search the knowledge base.
- Persist `chroma_store/` to Google Drive (`drive.mount`) so your index survives across Colab sessions.
